# CERDOS_25 — Pig Counting Diagnostics and Threshold Sensitivity
**Runtime: CPU (GPU not required).**  
**Purpose:** Publication-oriented descriptive audit of YOLO26 count errors using the frozen COCO predictions from CERDOS_23 and the image-level audit from CERDOS_24.

**Scientific safeguard:** Test-set threshold sweeps are **exploratory diagnostics only**. They must NOT be used to select or optimize an operational threshold. The reference counting threshold remains **0.25**; any revised threshold requires a separate validation set and prospective confirmation. High-overlap predicted masks are diagnostic candidates, not automatically duplicate animals.

**Figures:** English labels, color-blind-safe palette, 300/600 dpi PNG + vector PDF/SVG. **Tables:** Excel. No model training or inference.

**Version 1.1 (corrected):** fixed-threshold audit and pandas-compatible summary calculations.


In [ ]:
from google.colab import drive
from pathlib import Path
import os, zipfile, json, io, hashlib
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
from IPython.display import display

drive.mount('/content/drive')
ROOT=Path('/content/drive/MyDrive/Proyecto de investigación: Visión Artificial y Deep Learning en Cerdos 🐖')
OUT=ROOT/'03_Resultados'/'CERDOS_25_Count_Diagnostics'
OUT.mkdir(parents=True,exist_ok=True)
FIG=OUT/'Figures'; FIG.mkdir(exist_ok=True)
print('OUTPUT:',OUT)
print('RUNTIME: CPU; no GPU needed')


In [ ]:
# Find archived inputs anywhere inside the project, with Colab upload fallback.
from google.colab import files

def acquire(filename):
    hits=list(ROOT.rglob(filename)) if ROOT.exists() else []
    if not hits:
        print(f'Please upload {filename}')
        uploaded=files.upload()
        if filename not in uploaded: raise FileNotFoundError(filename)
        return Path('/content')/filename
    print('Found:',hits[0])
    return hits[0]

p23=acquire('CERDOS_23_YOLO26_Evaluacion_Resultados.zip')
p24=acquire('CERDOS_24_Resultados.zip')
with zipfile.ZipFile(p24) as z:
    book=pd.ExcelFile(io.BytesIO(z.read('CERDOS_24_Auditoria_Errores_YOLO26.xlsx')))
    images=pd.read_excel(book,'Por_Imagen')
    overlap=pd.read_excel(book,'Solapes_IoU_070')
with zipfile.ZipFile(p23) as z:
    protocol=json.loads(z.read('CERDOS_23_Protocolo.json'))
    preds={seed:json.loads(z.read(f'CERDOS_23_Pred_seed_{seed}.json')) for seed in [2026,2027,2028]}
print('Images:',images.shape,'Predictions:',{k:len(v) for k,v in preds.items()})
assert images.groupby('semilla')['image_id'].nunique().eq(373).all(), 'Unexpected number of images'
assert set(images['semilla'])=={2026,2027,2028}


## Counting definitions
A prediction at threshold *t* contributes one instance if `score >= t`. No class/shape filters are introduced. Original COCO prediction NMS parameters are already baked into the archived predictions. Therefore a confidence sweep **cannot recover detections filtered out earlier** and is not equivalent to rerunning YOLO at new NMS thresholds.

In [ ]:
# Construct count table at reference threshold and exploratory confidence levels.
THRESHOLDS=np.round(np.arange(0.05,0.951,0.05),2)
if 0.25 not in THRESHOLDS: THRESHOLDS=np.sort(np.r_[THRESHOLDS,0.25])
reference=images[['semilla','image_id','fecha','imagen','gt_conteo','pred_conteo_025']].copy()
records=[]
for seed,items in preds.items():
    p=pd.DataFrame([(int(v['image_id']),float(v['score'])) for v in items],columns=['image_id','score'])
    for t in THRESHOLDS:
        counts=p.loc[p.score>=t].groupby('image_id').size()
        part=reference.loc[reference.semilla==seed].copy()
        part['threshold']=float(t)
        part['pred_count']=part['image_id'].map(counts).fillna(0).astype(int)
        part['signed_error']=part.pred_count-part.gt_conteo
        part['absolute_error']=part.signed_error.abs()
        records.append(part)
sweep=pd.concat(records,ignore_index=True)
check=sweep[np.isclose(sweep.threshold,0.25)]
# Explicitly compare the fixed threshold with the independently archived reference counts.
expected=reference.set_index(['semilla','image_id'])['pred_conteo_025']
actual=check.set_index(['semilla','image_id'])['pred_count']
assert actual.index.is_unique and expected.index.is_unique, 'Duplicate image keys in inputs.'
assert actual.sort_index().equals(expected.reindex(actual.index).sort_index().astype(int)), \
    'Count mismatch at the prespecified threshold; check archived predictions.'
print('Reference count audit passed')


In [ ]:
def stats(g):
    return pd.Series({'n_images':len(g),'MAE':g.absolute_error.mean(),
                      'RMSE':np.sqrt(np.mean(g.signed_error**2)),
                      'bias':g.signed_error.mean(),
                      'exact_match_percent':100*(g.signed_error==0).mean(),
                      'overcount_percent':100*(g.signed_error>0).mean()})
summary=sweep.groupby(['semilla','threshold']).apply(stats).reset_index()
bydate=check.groupby(['semilla','fecha']).apply(stats).reset_index()
# Define errors by the prior fixed threshold only; scientific transparency for exploratory test analysis.
critical=check.sort_values(['absolute_error','semilla'],ascending=[False,True]).head(60)
# Potential overlap relation: descriptive only, not evidence of duplicate labels.
ov=overlap.groupby(['semilla','image_id']).size().rename('high_overlap_pairs').reset_index()
overlap_assoc=check.merge(ov,on=['semilla','image_id'],how='left')
overlap_assoc['high_overlap_pairs']=overlap_assoc.high_overlap_pairs.fillna(0).astype(int)
overlap_assoc['any_high_overlap']=overlap_assoc.high_overlap_pairs>0
assoc=overlap_assoc.groupby(['semilla','any_high_overlap']).apply(stats).reset_index()
display(summary[np.isclose(summary.threshold,0.25)],bydate,assoc)


In [ ]:
# Publication-ready visual style; 600 dpi PNG + SVG + PDF.
plt.rcParams.update({'font.family':'DejaVu Sans','font.size':11,'axes.labelsize':12,
 'axes.titlesize':13,'axes.spines.top':False,'axes.spines.right':False,
 'savefig.bbox':'tight','axes.grid':False,'legend.frameon':False})
COL={2026:'#0072B2',2027:'#D55E00',2028:'#009E73'}
def export(fig,name):
    for ext in ['png','pdf','svg']:
        fig.savefig(FIG/f'{name}.{ext}',dpi=600 if ext=='png' else None,facecolor='white')
    plt.show();plt.close(fig)
fig,ax=plt.subplots(figsize=(7.2,4.5))
for seed,d in summary.groupby('semilla'):
    ax.plot(d.threshold,d.MAE,marker='o',markersize=3,lw=1.8,color=COL[seed],label=f'Seed {seed}')
ax.axvline(.25,color='0.35',ls='--',lw=1,label='Prespecified threshold (0.25)')
ax.set(xlabel='Confidence threshold',ylabel='Mean absolute count error (pigs/image)',title='Exploratory test-set threshold sensitivity')
ax.legend(ncol=2,fontsize=9);ax.grid(axis='y',alpha=.2)
export(fig,'Fig25_01_Test_Threshold_Sensitivity_EXPLORATORY')

fig,ax=plt.subplots(figsize=(7.3,4.4))
for seed,d in bydate.groupby('semilla'):
    ax.plot(d.fecha.astype(str),d.MAE,marker='o',lw=2,color=COL[seed],label=f'Seed {seed}')
ax.set(xlabel='Acquisition date',ylabel='Mean absolute count error (pigs/image)',title='Counting error across acquisition dates (confidence = 0.25)')
ax.tick_params(axis='x',rotation=35);ax.grid(axis='y',alpha=.2);ax.legend()
export(fig,'Fig25_02_Count_MAE_by_Date')

fig,ax=plt.subplots(figsize=(6.5,5))
for seed,d in check.groupby('semilla'):
    ax.scatter(d.gt_conteo,d.pred_count,s=17,alpha=.35,color=COL[seed],label=f'Seed {seed}')
maximum=int(max(check.gt_conteo.max(),check.pred_count.max()))+1
ax.plot([0,maximum],[0,maximum],ls='--',color='black',lw=1,label='Perfect count')
ax.set(xlabel='Reference pig count',ylabel='Predicted pig count',title='Pig count agreement at confidence = 0.25',xlim=(0,maximum),ylim=(0,maximum))
ax.legend(fontsize=9);export(fig,'Fig25_03_Reference_vs_Predicted_Count')


In [ ]:
# Export auditable Excel and concise provenance report.
excel=OUT/'CERDOS_25_Count_Diagnostics.xlsx'
with pd.ExcelWriter(excel,engine='openpyxl') as w:
    summary.to_excel(w,'Threshold_Sensitivity',index=False)
    bydate.to_excel(w,'Reference_by_Date',index=False)
    check.to_excel(w,'Reference_Per_Image',index=False)
    critical.to_excel(w,'Critical_Images_Descriptive',index=False)
    assoc.to_excel(w,'High_Overlap_Comparison',index=False)
    overlap_assoc.to_excel(w,'Overlap_Per_Image',index=False)
    pd.DataFrame([{'item':'Evaluation dataset','value':'TEST — descriptive; no threshold selected'},
                 {'item':'Prespecified reference threshold','value':0.25},
                 {'item':'Inference device','value':'CPU'},
                 {'item':'COCO maxDets','value':str(protocol['prediction']['max_det'])},
                 {'item':'Prediction confidence floor','value':protocol['prediction']['conf']}]).to_excel(w,'Read_Me',index=False)
report={'experiment':'CERDOS_25','role':'TEST descriptive analysis only','reference_threshold':0.25,
 'seeds':[2026,2027,2028], 'data_sources':[p23.name,p24.name],
 'input_sha256':{p.name:hashlib.sha256(p.read_bytes()).hexdigest() for p in [p23,p24]},
 'limitation':'Threshold sensitivity on TEST is exploratory, not a basis for threshold choice; independent validation data required.'}
(OUT/'CERDOS_25_Provenance.json').write_text(json.dumps(report,indent=2),encoding='utf-8')
archive=OUT/'CERDOS_25_Resultados.zip'
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
    for p in [excel,OUT/'CERDOS_25_Provenance.json',*FIG.glob('*')]:z.write(p,p.relative_to(OUT))
print('COMPLETED:',archive)
print('Excel:',excel)


In [ ]:

from google.colab import files
from pathlib import Path

zip_path = Path(OUT) / "CERDOS_25_Resultados.zip"

if zip_path.exists():
    print("Downloading:", zip_path.name)
    files.download(str(zip_path))
else:
    print("ZIP not found. Check that the export cell ran successfully.")


### Interpretation for manuscript development
- **Do not report the lowest MAE on the test sweep as an optimized result.** Report the prespecified 0.25 reference results and describe the sweep as exploratory only.
- Mask overlap IoU > 0.70 flags cases for manual inspection; it does **not** establish false positives or duplicates without ground-truth instance matching.
- This experiment cannot isolate acquisition-condition causes without additional metadata.
- **Next required experiment:** tune confidence threshold on an untouched validation split, freeze it, and confirm on genuinely new external data.
